In [1]:
from pathlib import Path
import json
import math

import geopandas as gpd
import pandas as pd
import folium
import param
import panel as pn

pn.extension('floatpanel')

from folium.plugins import HeatMap
from branca.element import MacroElement, Template
from panel.reactive import ReactiveHTML

### LOCATION INFO CONNECTOR ###

import requests

from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs
from elevenlabs.play import play

### ffmpeg SUPPORT ###

import os
import sys

# Replace this with the actual path to the folder containing 'ffmpeg.exe'
ffmpeg_dir = r"C:\Users\dawns\ffmpeg\bin"

# Append the directory to the system PATH variable for this session only
os.environ["PATH"] += os.pathsep + ffmpeg_dir

### END OF ffmpeg SUPPORT ###


# ==========================================
# LOAD API KEYS
# ==========================================

load_dotenv()

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
XWEATHER_CLIENT_ID = os.getenv("XWEATHER_CLIENT_ID")
XWEATHER_CLIENT_SECRET = os.getenv("XWEATHER_CLIENT_SECRET")


# ==========================================
# ELEVENLABS SETUP
# ==========================================

client = ElevenLabs(
    api_key=ELEVENLABS_API_KEY
)


# ==========================================
# GET LIVE WEATHER FROM XWEATHER
# ==========================================

def get_weather(location):

    url = f"https://data.api.xweather.com/conditions/{location}"

    params = {
        "client_id": XWEATHER_CLIENT_ID,
        "client_secret": XWEATHER_CLIENT_SECRET
    }

    print('params done')

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    print('req done')

    response.raise_for_status()

    data = response.json()

    if not data.get("success"):
        return None

    result = data["response"][0]
    conditions = result["periods"][0]

    print('rest done')

    return {
        "weather": conditions["weatherPrimary"],
        "temperature": conditions["tempC"],
        "precipitation": conditions["precipMM"]
    }


# ==========================================
# DETERMINE WEATHER IMPACT
# ==========================================

def get_weather_impact(weather, temperature, precipitation):

    condition = weather.lower()

    # Heavy snow
    if "snow" in condition and precipitation >= 5:
        return "Snow may cause significant delays and slower travel."

    # Light / normal snow
    elif "snow" in condition:
        return "Snow may cause minor delays and slippery conditions."

    # Ice / freezing rain
    elif (
        "freezing" in condition
        or "ice" in condition
        or "sleet" in condition
    ):
        return "Icy conditions may cause delays and hazardous travel."

    # Thunderstorms
    elif (
        "thunderstorm" in condition
        or "thunder" in condition
    ):
        return "Thunderstorms may cause delays and slower travel."

    # Heavy rain
    elif "rain" in condition and precipitation >= 5:
        return "Heavy rain may cause delays and slower travel."

    # Light / normal rain
    elif "rain" in condition or precipitation > 0:
        return "Minor delays are possible due to rain."

    # Very cold
    elif temperature <= 0:
        return "Cold conditions are present, but no major weather delays are expected."

    # Normal weather
    else:
        return "No major weather delays are expected."

### END OF LOCATION INFO CONNECTOR ###


### ELEVENLABS ###

def speak(message):

    print("\nVOICE MESSAGE:")
    print(message)

    audio = client.text_to_speech.convert(
        text=message,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_v4",
        output_format="mp3_44100_128"
    )

    play(audio)

### END OF ELEVENLABS ###



# ============================================================
# CONFIGURATIONS
# ============================================================

DATA_DIR = Path("data")

# Statistics Canada population table
POPULATION_FILENAME = "98100015.csv"

# Map
MAP_CENTER = [56.0, -106.0]
INITIAL_ZOOM = 4

# Transit heatmap
TRANSIT_RADIUS = 10
TRANSIT_BLUR = 15

# Population heatmap
POPULATION_RADIUS = 15
POPULATION_BLUR = 20


# ============================================================
# MAP CLICK BRIDGE
# ============================================================

class MapClickBridge(ReactiveHTML):
    """
    Small Panel <-> JavaScript bridge.

    JavaScript receives map clicks through window.postMessage()
    and writes the coordinates into the synced `value` parameter.
    """

    value = param.String(default="")

    _template = "<div></div>"

    _scripts = {
        "render": """
            state.handler = (event) => {
                if (event.data?.type === "map_click") {
                    data.value = JSON.stringify(event.data);
                }
            };

            window.addEventListener(
                "message",
                state.handler
            );
        """,

        "remove": """
            window.removeEventListener(
                "message",
                state.handler
            );
        """
    }


def add_click_bridge(m):
    bridge = MacroElement()
    bridge._template = Template("""
    {% macro script(this, kwargs) %}
    var map = {{this._parent.get_name()}};
    var selectedMarker = null;

    map.on('click', function(e) {
        var lat = e.latlng.lat;
        var lon = e.latlng.lng;

        // Create the marker on the first click
        if (selectedMarker === null) {
            selectedMarker = L.marker(e.latlng).addTo(map);
            selectedMarker.bindPopup(
                "Latitude: " + lat.toFixed(6) +
                "<br>Longitude: " + lon.toFixed(6)
            );
        } else {
            // Move the existing marker on subsequent clicks
            selectedMarker.setLatLng(e.latlng);
            selectedMarker.setPopupContent(
                "Latitude: " + lat.toFixed(6) +
                "<br>Longitude: " + lon.toFixed(6)
            );
        }

        // Send the coordinates to Python as before
        window.parent.postMessage({
            type: 'map_click',
            lat: lat,
            lon: lon
        }, '*');
    });
    {% endmacro %}
    """)
    m.add_child(bridge)


# ============================================================
# FILE DETECTION
# ============================================================

def find_population_file():
    """
    Find the Statistics Canada population CSV.

    Expected filename:
        98100015.csv
    """

    population_file = (
        DATA_DIR / POPULATION_FILENAME
    )

    if not population_file.exists():

        raise FileNotFoundError(
            f"""
Could not find the population file:

    {population_file.resolve()}

Make sure 98100015.csv is inside:

    {DATA_DIR.resolve()}
"""
        )

    print(
        f"Population file: {population_file}"
    )

    return population_file


def find_population_boundary_file():
    """
    Automatically find the 2021 Statistics Canada
    dissemination-area boundary shapefile.
    """

    candidates = list(
        DATA_DIR.glob("lda_*21*.shp")
    )

    if not candidates:

        raise FileNotFoundError(
            f"""
Could not find a 2021 dissemination-area
boundary shapefile in:

    {DATA_DIR.resolve()}

Expected something similar to:

    lda_000b21a_e.shp
"""
        )

    # Prefer English boundary files if multiple
    # versions are present.
    english_candidates = [
        file
        for file in candidates
        if file.stem.lower().endswith("_e")
    ]

    if english_candidates:
        candidates = english_candidates

    if len(candidates) > 1:

        print(
            "\nMultiple possible DA boundary files found:"
        )

        for file in candidates:
            print(
                f"  {file.name}"
            )

        print(
            f"\nUsing: {candidates[0].name}"
        )

    boundary_file = candidates[0]

    print(
        f"Boundary file: {boundary_file}"
    )

    return boundary_file


def find_transit_file():
    """
    Automatically find the transit GeoPackage.
    """

    gpkg_files = list(
        DATA_DIR.glob("*.gpkg")
    )

    if not gpkg_files:

        raise FileNotFoundError(
            f"""
Could not find a GeoPackage in:

    {DATA_DIR.resolve()}
"""
        )

    if len(gpkg_files) > 1:

        print(
            "\nMultiple GeoPackages found:"
        )

        for file in gpkg_files:
            print(
                f"  {file.name}"
            )

        print(
            f"\nUsing: {gpkg_files[0].name}"
        )

    transit_file = gpkg_files[0]

    print(
        f"Transit file: {transit_file}"
    )

    return transit_file


# ============================================================
# TRANSIT DATA
# ============================================================

def find_transit_layer(transit_file):
    """
    Find the transit stops layer.

    Prefer a layer named 'stops'. Otherwise,
    use the first Point layer.
    """

    layers = gpd.list_layers(
        transit_file
    )

    print("\nTransit GeoPackage layers:")

    point_layers = []

    for i in range(len(layers)):

        layer_name = layers.iloc[i]["name"]
        geometry_type = layers.iloc[i]["geometry_type"]

        print(
            f"  {layer_name} "
            f"({geometry_type})"
        )

        if (
            str(geometry_type).upper()
            == "POINT"
        ):
            point_layers.append(
                layer_name
            )

    if not point_layers:

        raise RuntimeError(
            """
Could not find a Point layer
in the transit GeoPackage.
"""
        )

    # Prefer "stops"
    for layer in point_layers:

        if str(layer).lower() == "stops":

            return layer

    # Otherwise use first point layer
    return point_layers[0]


def load_transit_stops():
    """
    Load transit stop points.
    """

    print(
        "\nLoading transit stops..."
    )

    transit_file = find_transit_file()

    transit_layer = find_transit_layer(
        transit_file
    )

    print(
        f"\nUsing transit layer: "
        f"{transit_layer}"
    )

    stops = gpd.read_file(
        transit_file,
        layer=transit_layer
    )

    print(
        f"Loaded {len(stops):,} "
        f"transit stops."
    )

    if stops.crs is None:

        raise RuntimeError(
            "Transit data has no CRS."
        )

    stops = stops.to_crs(
        "EPSG:4326"
    )

    stops = stops[
        stops.geometry.notna()
        &
        ~stops.geometry.is_empty
        &
        (
            stops.geometry.geom_type
            == "Point"
        )
    ].copy()

    print(
        f"Valid transit stops: "
        f"{len(stops):,}"
    )

    return stops


def aggregate_transit_stops(stops):
    """
    Aggregate transit stops into a grid.
    """

    print(
        "Aggregating transit stops..."
    )

    stops = stops.copy()

    stops["longitude"] = (
        stops.geometry.x
    )

    stops["latitude"] = (
        stops.geometry.y
    )

    GRID_SIZE = 0.02

    stops["grid_lon"] = (
        stops["longitude"]
        / GRID_SIZE
    ).round() * GRID_SIZE

    stops["grid_lat"] = (
        stops["latitude"]
        / GRID_SIZE
    ).round() * GRID_SIZE

    grid = (
        stops
        .groupby(
            [
                "grid_lat",
                "grid_lon"
            ]
        )
        .size()
        .reset_index(
            name="weight"
        )
    )

    print(
        f"Reduced {len(stops):,} "
        f"stops to "
        f"{len(grid):,} "
        f"heatmap cells."
    )

    max_weight = (
        grid["weight"].max()
    )

    if max_weight > 0:

        grid["weight"] = (
            grid["weight"]
            / max_weight
        )

    return grid


# ============================================================
# POPULATION DATA
# ============================================================

def load_population_data():
    """
    Load Statistics Canada table 98-10-0015-02.
    """

    print(
        "\nLoading census population data..."
    )

    population_file = (
        find_population_file()
    )

    df = pd.read_csv(
        population_file,
        encoding="utf-8-sig"
    )

    print(
        f"Loaded {len(df):,} "
        f"census records."
    )

    print(
        "\nPopulation CSV columns:"
    )

    for column in df.columns:

        print(
            f"  {column!r}"
        )

    if "GEO" not in df.columns:

        raise RuntimeError(
            """
The population CSV does not contain
a GEO column.
"""
        )

    # --------------------------------------------------------
    # Find population column
    # --------------------------------------------------------

    population_column = None

    for column in df.columns:

        column_string = str(column)

        if (
            "Population, 2021 [1]"
            in column_string
        ):

            population_column = column

            break

    if population_column is None:

        raise RuntimeError(
            """
Could not find the 2021 population
column in 98100015.csv.
"""
        )

    # --------------------------------------------------------
    # Find population density column
    # --------------------------------------------------------

    density_column = None

    for column in df.columns:

        column_string = str(column)

        if (
            "Population density per square kilometre"
            in column_string
            and
            "[5]" in column_string
        ):

            density_column = column

            break

    if density_column is None:

        raise RuntimeError(
            """
Could not find the 2021 population-density
column in 98100015.csv.
"""
        )

    print(
        "\nUsing columns:"
    )

    print(
        "  DAUID: GEO"
    )

    print(
        f"  Population: "
        f"{population_column}"
    )

    print(
        f"  Population density: "
        f"{density_column}"
    )

    population = pd.DataFrame()

    population["DAUID"] = (
        df["GEO"]
        .astype(str)
        .str.strip()
    )

    population["population"] = (
        pd.to_numeric(
            df[population_column],
            errors="coerce"
        )
    )

    population["population_density"] = (
        pd.to_numeric(
            df[density_column],
            errors="coerce"
        )
    )

    population = population[
        population["DAUID"]
        .str.fullmatch(
            r"\d{8}"
        )
    ].copy()

    print(
        f"\nFound "
        f"{len(population):,} "
        f"8-digit geographic records."
    )

    population = population.dropna(
        subset=[
            "population",
            "population_density"
        ]
    )

    population = (
        population
        .drop_duplicates(
            subset="DAUID"
        )
    )

    print(
        f"Valid DA population records: "
        f"{len(population):,}"
    )

    if len(population) > 0:

        minimum = (
            population[
                "population_density"
            ].min()
        )

        maximum = (
            population[
                "population_density"
            ].max()
        )

        print(
            f"Population density range: "
            f"{minimum:,.2f} - "
            f"{maximum:,.2f} "
            f"people/km²"
        )

    return population


# ============================================================
# POPULATION BOUNDARIES
# ============================================================

def load_population_boundaries():
    """
    Load the 2021 Canadian dissemination-area
    boundary shapefile.
    """

    print(
        "\nLoading census boundaries..."
    )

    boundary_file = (
        find_population_boundary_file()
    )

    boundaries = gpd.read_file(
        boundary_file
    )

    print(
        f"Loaded {len(boundaries):,} "
        f"dissemination areas."
    )

    if "DAUID" not in boundaries.columns:

        print(
            "\nBoundary file columns:"
        )

        for column in boundaries.columns:

            print(
                f"  {column}"
            )

        raise RuntimeError(
            """
The dissemination-area boundary
file does not contain a DAUID column.
"""
        )

    boundaries["DAUID"] = (
        boundaries["DAUID"]
        .astype(str)
        .str.strip()
    )

    boundaries = boundaries.to_crs(
        "EPSG:4326"
    )

    return boundaries


# ============================================================
# JOIN POPULATION TO BOUNDARIES
# ============================================================

def join_population_data(
    boundaries,
    population
):
    """
    Join population density to the
    dissemination-area polygons.
    """

    print(
        "\nJoining census data to boundaries..."
    )

    joined = boundaries.merge(
        population,
        on="DAUID",
        how="left"
    )

    matched = (
        joined[
            "population_density"
        ]
        .notna()
        .sum()
    )

    total = len(joined)

    percentage = (
        matched / total * 100
        if total > 0
        else 0
    )

    print(
        f"Matched {matched:,} / "
        f"{total:,} dissemination areas "
        f"({percentage:.1f}%)."
    )

    return joined


# ============================================================
# POPULATION HEATMAP
# ============================================================

def create_population_heatmap(
    population_boundaries
):
    """
    Create heatmap points from dissemination areas.
    """

    print(
        "\nCreating population density heatmap..."
    )

    data = population_boundaries[
        population_boundaries[
            "population_density"
        ].notna()
        &
        population_boundaries.geometry.notna()
        &
        ~population_boundaries.geometry.is_empty
    ].copy()

    print(
        f"Using {len(data):,} "
        f"dissemination areas."
    )

    if len(data) == 0:

        raise RuntimeError(
            """
No dissemination areas have valid
population-density data.

This usually means the DAUIDs in the
population CSV do not match the DAUIDs
in the boundary file.
"""
        )

    points = (
        data.geometry
        .representative_point()
    )

    density = (
        data["population_density"]
        .clip(lower=0)
    )

    transformed_density = (
        density.apply(
            math.log1p
        )
    )

    minimum = (
        transformed_density.min()
    )

    maximum = (
        transformed_density.max()
    )

    if maximum > minimum:

        weights = (
            transformed_density
            - minimum
        ) / (
            maximum - minimum
        )

    else:

        weights = (
            transformed_density
            * 0
            + 1
        )

    heat_data = []

    for point, weight in zip(
        points,
        weights
    ):

        if point is None:
            continue

        if point.is_empty:
            continue

        if pd.isna(weight):
            continue

        heat_data.append(
            [
                point.y,
                point.x,
                float(weight)
            ]
        )

    print(
        f"Created {len(heat_data):,} "
        f"population heatmap points."
    )

    return heat_data


# ============================================================
# CREATE MAP
# ============================================================

def create_map(
    transit_grid,
    population_heatmap
):
    """
    Create the Folium map containing both heatmaps.
    """

    print(
        "\nCreating map..."
    )

    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=INITIAL_ZOOM,
        tiles=None,
        control_scale=True
    )

    # --------------------------------------------------------
    # OpenStreetMap
    # --------------------------------------------------------

    folium.TileLayer(
        tiles=(
            "https://{s}.tile.openstreetmap.org/"
            "{z}/{x}/{y}.png"
        ),
        attr=(
            "© OpenStreetMap contributors"
        ),
        name="OpenStreetMap",
        overlay=False,
        control=True
    ).add_to(m)

    # ========================================================
    # POPULATION HEATMAP
    # ========================================================

    print(
        "Creating population density heatmap..."
    )

    population_group = (
        folium.FeatureGroup(
            name="Population Density",
            show=True
        )
    )

    HeatMap(
        population_heatmap,

        radius=POPULATION_RADIUS,

        blur=POPULATION_BLUR,

        min_opacity=0.25,

        gradient={
            0.2: "purple",
            0.4: "magenta",
            0.6: "red",
            0.8: "orange",
            1.0: "yellow"
        }
    ).add_to(
        population_group
    )

    population_group.add_to(m)
    
    # ========================================================
    # TRANSIT HEATMAP
    # ========================================================

    print(
        "Creating transit heatmap..."
    )

    transit_group = (
        folium.FeatureGroup(
            name="Transit Stop Density",
            show=True
        )
    )

    HeatMap(
        transit_grid[
            [
                "grid_lat",
                "grid_lon",
                "weight"
            ]
        ].values.tolist(),

        radius=TRANSIT_RADIUS,

        blur=TRANSIT_BLUR,

        min_opacity=0.25,

        gradient={
            0.2: "blue",
            0.4: "cyan",
            0.6: "lime",
            0.8: "yellow",
            1.0: "red"
        }
    ).add_to(
        transit_group
    )

    transit_group.add_to(m)

    # ========================================================
    # LAYER CONTROL
    # ========================================================

    folium.LayerControl(
        collapsed=False
    ).add_to(m)

    # ========================================================
    # MAP CLICK BRIDGE
    # ========================================================

    add_click_bridge(m)

    return m


# ============================================================
# PANEL APPLICATION
# ============================================================

print(str(get_weather))

def build_app():

    print(
        "\n"
        + "=" * 60
    )

    print(
        "CANADIAN TRANSIT + "
        "POPULATION DENSITY MAP"
    )

    print(
        "=" * 60
        + "\n"
    )

    # ========================================================
    # TRANSIT
    # ========================================================

    stops = load_transit_stops()

    transit_grid = (
        aggregate_transit_stops(
            stops
        )
    )

    # ========================================================
    # POPULATION
    # ========================================================

    population = (
        load_population_data()
    )

    boundaries = (
        load_population_boundaries()
    )

    population_boundaries = (
        join_population_data(
            boundaries,
            population
        )
    )

    population_heatmap = (
        create_population_heatmap(
            population_boundaries
        )
    )

    # ========================================================
    # MAP
    # ========================================================

    transit_map = create_map(
        transit_grid,
        population_heatmap
    )

    # ========================================================
    # CLICK BRIDGE
    # ========================================================

    bridge = MapClickBridge()

    clicked_location = pn.pane.Markdown(
        "### Click somewhere on the map"
    )

    def tss_click(event): 
        message = (
            "### Selected Location\n"
            f"**Latitude is ** {lat:.26f}  \n"
            f"**Longitude: is ** {lon:.6f}  \n"
            f"**Temperature is ** {weather['temperature']}  degrees Celsuis\n"
            f"**Precipitation is ** {weather['precipitation']}  millimeters\n"
            f"**Impact of the weather is that ** {get_weather_impact(weather['weather'], weather['temperature'], weather['precipitation'])}"
        )
        speak(message)
    tts_btn = pn.widgets.Button(label='Text to Speech', color='primary')
    tts_btn.on_click(tss_click)
    
    clicked_popup = pn.layout.FloatPanel(
        pn.Column(clicked_location, tts_btn), name="Location Information", margin=20, status='smallified'
    )

    def handle_click(event):

        if not event.new:
            return

        try:

            data = json.loads(
                event.new
            )

            lat = data["lat"]
            lon = data["lon"]

            print(f"{lat:.2f}", f"{lon:.2f}")

            weather = get_weather(f"{lat:.2f},{lon:.2f}")

            clicked_location.object = (
                "### Selected Location\n"
                f"**Latitude:** {lat:.6f}  \n"
                f"**Longitude:** {lon:.6f}  \n"
                f"**Temperature:** {weather['temperature']}  \n"
                f"**Precipitation:** {weather['precipitation']}  \n"
                f"**Impact:** {get_weather_impact(weather['weather'], weather['temperature'], weather['precipitation'])}"
            )
            clicked_popup[0].pop(1)
            def tss_click(event): 
                message = (
                    "### Selected Location\n"
                    f"**Latitude is ** {lat:.2f}  \n"
                    f"**Longitude: is ** {lon:.2f}  \n"
                    f"**Temperature is ** {weather['temperature']}  degrees Celsuis\n"
                    f"**Precipitation is ** {weather['precipitation']}  millimeters\n"
                    f"**Impact of the weather is that ** {get_weather_impact(weather['weather'], weather['temperature'], weather['precipitation'])}"
                )
                speak(message)
            tts_btn = pn.widgets.Button(label='Text to Speech', color='primary')
            tts_btn.on_click(tss_click)
            clicked_popup[0].append(tts_btn)
            clicked_popup.status = 'smallified'

        except (
            json.JSONDecodeError,
            KeyError,
            TypeError
        ):

            pass

    bridge.param.watch(
        handle_click,
        "value"
    )

    # ========================================================
    # PANEL MAP
    # ========================================================

    map_pane = pn.pane.plot.Folium(
        transit_map,
        height=750,
        sizing_mode="stretch_width"
    )

    title = pn.pane.Markdown(
        "# The Bus Router"
    )

    description = pn.pane.Markdown(
        """This is **The Bus Router**, planning the bus routes of tomorrow. When considering where to build 
        transit extensions, you need to take into account population density, pre-existing bus routes, and 
        even weather patterns. The Bus Router gives you a clean and effective way to visualize all of these 
        aspects you need to be considering. Try clicking on the map to see live details regarding the relevant 
        conditions an area is experiencing. """, max_width = 960
    )

    app = pn.Column(
        pn.Row(pn.Column(title, description), clicked_popup), 
        map_pane,
        bridge,
        sizing_mode="stretch_width"
    )
    return app



pn.extension()

app = build_app()


if __name__ == "__main__":

    pn.serve(
        app,
        title=(
            "Canadian Transit & "
            "Population Density"
        ),
        show=True
    )

<function get_weather at 0x000001D4ADDB4FE0>



CANADIAN TRANSIT + POPULATION DENSITY MAP


Loading transit stops...
Transit file: data\stops_and_routes.gpkg

Transit GeoPackage layers:
  stops (Point)
  shapes (LineString)

Using transit layer: stops
Loaded 132,896 transit stops.


C:\Users\dawns\AppData\Local\Temp\ipykernel_19624\61140552.py:510: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  stops.geometry.notna()


Valid transit stops: 132,887
Aggregating transit stops...
Reduced 132,887 stops to 6,696 heatmap cells.

Loading census population data...
Population file: data\98100015.csv
Loaded 63,404 census records.

Population CSV columns:
  'REF_DATE'
  'GEO'
  'DGUID'
  'Coordinate'
  'Population and dwelling counts (5): Population, 2021 [1]'
  'Symbols'
  'Population and dwelling counts (5): Total private dwellings, 2021 [2]'
  'Symbols.1'
  'Population and dwelling counts (5): Private dwellings occupied by usual residents, 2021 [3]'
  'Symbols.2'
  'Population and dwelling counts (5): Land area in square kilometres, 2021 [4]'
  'Symbols.3'
  'Population and dwelling counts (5): Population density per square kilometre, 2021 [5]'
  'Symbols.4'

Using columns:
  DAUID: GEO
  Population: Population and dwelling counts (5): Population, 2021 [1]
  Population density: Population and dwelling counts (5): Population density per square kilometre, 2021 [5]

Found 57,936 8-digit geographic records.
Valid


VOICE MESSAGE:
### Selected Location
**Latitude is ** 49.95  
**Longitude: is ** -120.42  
**Temperature is ** 6.77  degrees Celsuis
**Precipitation is ** 0  millimeters
**Impact of the weather is that ** No major weather delays are expected.
